# 03 — Classification: Logistic Regression & KNN
**Dataset:** Credit Card Fraud Detection — të dhëna të parapërpunuara nga `02_preprocessing.ipynb`

Qëllimi: Trajnojmë dhe vlerësojmë dy klasifikues — **Logistic Regression** (linear) dhe **K-Nearest Neighbors** (distance-based) — me hyperparameter tuning për secilin.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

RANDOM_STATE = 42

In [ ]:
X_train       = np.load('../data/processed/X_train.npy')
X_test        = np.load('../data/processed/X_test.npy')
y_train       = np.load('../data/processed/y_train.npy')
y_test        = np.load('../data/processed/y_test.npy')
feature_names = np.load('../data/processed/feature_names.npy', allow_pickle=True)

print("=" * 50)
print("TË DHËNAT E NGARKUARA")
print("=" * 50)
print(f"X_train : {X_train.shape}  (SMOTE-balanced)")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}  | Fraud: {(y_train==1).sum():,}  Legjitime: {(y_train==0).sum():,}")
print(f"y_test  : {y_test.shape}   | Fraud: {(y_test==1).sum():,}   Legjitime: {(y_test==0).sum():,}")
print(f"\nFeatures ({len(feature_names)}): {list(feature_names)}")

## 1. Logistic Regression
Klasifikues linear që modelon probabilitetin e klasës duke përdorur funksionin sigmoid. Hyperparameter kryesor: **C** (inversi i regularizimit) — vlera e vogël = regularizim i fortë, vlera e madhe = regularizim i dobët.

In [ ]:
param_grid_lr = {
    'C'      : [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver' : ['liblinear']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_lr = GridSearchCV(
    LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    param_grid_lr,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_lr.fit(X_train, y_train)

print("=" * 50)
print("REZULTATET E GRIDSEARCHCV — LOGISTIC REGRESSION")
print("=" * 50)
print(f"Parametrat më të mirë : {grid_lr.best_params_}")
print(f"F1-score më i mirë (CV): {grid_lr.best_score_:.4f}")

In [ ]:
results_lr = pd.DataFrame(grid_lr.cv_results_)
results_lr = results_lr[['param_C', 'param_penalty', 'mean_test_score', 'std_test_score']]
results_lr = results_lr.sort_values('mean_test_score', ascending=False)
results_lr.columns = ['C', 'Penalty', 'F1 mesatar (CV)', 'Std']
print("Top 8 kombinime:")
print(results_lr.head(8).round(4).to_string(index=False))

### 1.2 Vlerësimi i Logistic Regression
Vlerësojmë modelin më të mirë në **test set** me metrika: Accuracy, Precision, Recall, F1, ROC-AUC dhe Confusion Matrix.

In [ ]:
best_lr   = grid_lr.best_estimator_
y_pred_lr = best_lr.predict(X_test)
y_prob_lr = best_lr.predict_proba(X_test)[:, 1]

acc_lr  = accuracy_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr)
rec_lr  = recall_score(y_test, y_pred_lr)
f1_lr   = f1_score(y_test, y_pred_lr)
auc_lr  = roc_auc_score(y_test, y_prob_lr)

print("=" * 50)
print("METRIKAT — LOGISTIC REGRESSION (Test Set)")
print("=" * 50)
print(f"Accuracy  : {acc_lr:.4f}")
print(f"Precision : {prec_lr:.4f}")
print(f"Recall    : {rec_lr:.4f}")
print(f"F1-score  : {f1_lr:.4f}")
print(f"ROC-AUC   : {auc_lr:.4f}")
print(f"\n{classification_report(y_test, y_pred_lr, target_names=['Legjitime', 'Mashtruese'])}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Legjitime', 'Mashtruese'],
            yticklabels=['Legjitime', 'Mashtruese'])
axes[0].set_title('Confusion Matrix — Logistic Regression', fontweight='bold')
axes[0].set_ylabel('Aktuale')
axes[0].set_xlabel('Të parashikuara')

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
axes[1].plot(fpr_lr, tpr_lr, color='steelblue', linewidth=2,
             label=f'Logistic Regression (AUC = {auc_lr:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
axes[1].set_title('ROC Curve — Logistic Regression', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.tight_layout()
plt.savefig('../images/lr_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. K-Nearest Neighbors (KNN)
Klasifikues distance-based që klasifikon një pikë bazuar në K fqinjët më të afërt. Hyperparametrat kryesorë: **n_neighbors** (K), **metric** (distanca), **weights** (uniform ose distance-weighted).

In [ ]:
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 11, 15, 21],
    'weights'    : ['uniform', 'distance'],
    'metric'     : ['euclidean', 'manhattan']
}

grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid_knn,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_knn.fit(X_train, y_train)

print("=" * 50)
print("REZULTATET E GRIDSEARCHCV — KNN")
print("=" * 50)
print(f"Parametrat më të mirë : {grid_knn.best_params_}")
print(f"F1-score më i mirë (CV): {grid_knn.best_score_:.4f}")

In [ ]:
results_knn = pd.DataFrame(grid_knn.cv_results_)
results_knn = results_knn[['param_n_neighbors', 'param_weights', 'param_metric',
                            'mean_test_score', 'std_test_score']]
results_knn = results_knn.sort_values('mean_test_score', ascending=False)
results_knn.columns = ['K', 'Weights', 'Metric', 'F1 mesatar (CV)', 'Std']
print("Top 8 kombinime:")
print(results_knn.head(8).round(4).to_string(index=False))

### 2.2 Vlerësimi i KNN
Vlerësojmë modelin më të mirë KNN në test set me të njëjtat metrika si Logistic Regression.

In [ ]:
best_knn   = grid_knn.best_estimator_
y_pred_knn = best_knn.predict(X_test)
y_prob_knn = best_knn.predict_proba(X_test)[:, 1]

acc_knn  = accuracy_score(y_test, y_pred_knn)
prec_knn = precision_score(y_test, y_pred_knn)
rec_knn  = recall_score(y_test, y_pred_knn)
f1_knn   = f1_score(y_test, y_pred_knn)
auc_knn  = roc_auc_score(y_test, y_prob_knn)

print("=" * 50)
print("METRIKAT — KNN (Test Set)")
print("=" * 50)
print(f"Accuracy  : {acc_knn:.4f}")
print(f"Precision : {prec_knn:.4f}")
print(f"Recall    : {rec_knn:.4f}")
print(f"F1-score  : {f1_knn:.4f}")
print(f"ROC-AUC   : {auc_knn:.4f}")
print(f"\n{classification_report(y_test, y_pred_knn, target_names=['Legjitime', 'Mashtruese'])}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_knn = confusion_matrix(y_test, y_pred_knn)
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Oranges', ax=axes[0],
            xticklabels=['Legjitime', 'Mashtruese'],
            yticklabels=['Legjitime', 'Mashtruese'])
axes[0].set_title('Confusion Matrix — KNN', fontweight='bold')
axes[0].set_ylabel('Aktuale')
axes[0].set_xlabel('Të parashikuara')

fpr_knn, tpr_knn, _ = roc_curve(y_test, y_prob_knn)
axes[1].plot(fpr_knn, tpr_knn, color='tomato', linewidth=2,
             label=f'KNN (AUC = {auc_knn:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
axes[1].set_title('ROC Curve — KNN', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.tight_layout()
plt.savefig('../images/knn_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Krahasimi: Logistic Regression vs KNN
Krahasojmë të dy modelet me të gjitha metrikat për të identifikuar cili performon më mirë.

In [ ]:
comparison = pd.DataFrame({
    'Modeli'    : ['Logistic Regression', 'KNN'],
    'Accuracy'  : [acc_lr,  acc_knn],
    'Precision' : [prec_lr, prec_knn],
    'Recall'    : [rec_lr,  rec_knn],
    'F1-score'  : [f1_lr,   f1_knn],
    'ROC-AUC'   : [auc_lr,  auc_knn],
    'Best Params': [str(grid_lr.best_params_), str(grid_knn.best_params_)]
})

print("=" * 70)
print("TABELA KRAHASUESE — LOGISTIC REGRESSION vs KNN")
print("=" * 70)
print(comparison[['Modeli', 'Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC']].round(4).to_string(index=False))

best_model_name = comparison.loc[comparison['F1-score'].idxmax(), 'Modeli']
print(f"\nModeli më i mirë (F1): {best_model_name}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

metrics    = ['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC']
vals_lr    = [acc_lr,  prec_lr, rec_lr,  f1_lr,  auc_lr]
vals_knn   = [acc_knn, prec_knn, rec_knn, f1_knn, auc_knn]

x = np.arange(len(metrics))
w = 0.35
axes[0].bar(x - w/2, vals_lr,  w, label='Logistic Regression', color='steelblue', edgecolor='black', linewidth=0.5)
axes[0].bar(x + w/2, vals_knn, w, label='KNN',                  color='tomato',    edgecolor='black', linewidth=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].set_ylim(0, 1.1)
axes[0].set_title('Krahasimi i Metrikave', fontweight='bold')
axes[0].set_ylabel('Vlera')
axes[0].legend()
for i, (v1, v2) in enumerate(zip(vals_lr, vals_knn)):
    axes[0].text(i - w/2, v1 + 0.01, f'{v1:.3f}', ha='center', fontsize=8)
    axes[0].text(i + w/2, v2 + 0.01, f'{v2:.3f}', ha='center', fontsize=8)

axes[1].plot(fpr_lr,  tpr_lr,  color='steelblue', linewidth=2, label=f'Logistic Regression (AUC={auc_lr:.4f})')
axes[1].plot(fpr_knn, tpr_knn, color='tomato',    linewidth=2, label=f'KNN (AUC={auc_knn:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
axes[1].set_title('ROC Curves — Krahasim', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.suptitle('Logistic Regression vs KNN', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/lr_vs_knn.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Random Forest
Klasifikues ensemble bazuar ne pemë vendimi. Trajnon **N peme** ne subsets te ndryshme te te dhenave (bagging) dhe kombinon rezultatet me votim shumice.

**Avantazhet mbi modelet lineare:**
- Kapturon relacione jolineare dhe interaksione komplekse mes featureve
- I qendrueshem ndaj outlier-ave dhe noise-it
- Jep **feature importance** per interpretueshmeri

**Hyperparametrat kryesore:**  (numri i pemeve),  (thellesia), 

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

print("Importet per Random Forest dhe XGBoost u ngarkuan me sukses!")

In [ ]:
param_grid_rf = {
    "n_estimators" : [100, 200, 300],
    "max_depth"    : [10, 20, None],
    "min_samples_split": [2, 5],
    "class_weight" : ["balanced"]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid_rf,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_train, y_train)

print("=" * 50)
print("REZULTATET E GRIDSEARCHCV — RANDOM FOREST")
print("=" * 50)
print(f"Parametrat me te mire : {grid_rf.best_params_}")
print(f"F1-score me i mire (CV): {grid_rf.best_score_:.4f}")

In [ ]:
results_rf = pd.DataFrame(grid_rf.cv_results_)
results_rf = results_rf[["param_n_estimators", "param_max_depth",
                         "param_min_samples_split", "mean_test_score", "std_test_score"]]
results_rf = results_rf.sort_values("mean_test_score", ascending=False)
results_rf.columns = ["N Estimators", "Max Depth", "Min Samples Split", "F1 mesatar (CV)", "Std"]
print("Top 8 kombinime:")
print(results_rf.head(8).round(4).to_string(index=False))

### 3.2 Vlerësimi i Random Forest
Vlerësojmë modelin më të mirë në **test set** me të njëjtat metrika si modelet e mëparshme.

In [ ]:
best_rf   = grid_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]

acc_rf  = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf)
rec_rf  = recall_score(y_test, y_pred_rf)
f1_rf   = f1_score(y_test, y_pred_rf)
auc_rf  = roc_auc_score(y_test, y_prob_rf)

print("=" * 50)
print("METRIKAT — RANDOM FOREST (Test Set)")
print("=" * 50)
print(f"Accuracy  : {acc_rf:.4f}")
print(f"Precision : {prec_rf:.4f}")
print(f"Recall    : {rec_rf:.4f}")
print(f"F1-score  : {f1_rf:.4f}")
print(f"ROC-AUC   : {auc_rf:.4f}")
print(f"
{classification_report(y_test, y_pred_rf, target_names=['Legjitime', 'Mashtruese'])}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Greens", ax=axes[0],
            xticklabels=["Legjitime", "Mashtruese"],
            yticklabels=["Legjitime", "Mashtruese"])
axes[0].set_title("Confusion Matrix — Random Forest", fontweight="bold")
axes[0].set_ylabel("Aktuale")
axes[0].set_xlabel("Te parashikuara")

fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
axes[1].plot(fpr_rf, tpr_rf, color="seagreen", linewidth=2,
             label=f"Random Forest (AUC = {auc_rf:.4f})")
axes[1].plot([0, 1], [0, 1], "k--", linewidth=1, label="Random Classifier")
axes[1].set_title("ROC Curve — Random Forest", fontweight="bold")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend()

feat_imp = pd.Series(best_rf.feature_importances_, index=feature_names).sort_values(ascending=False).head(10)
axes[2].barh(feat_imp.index[::-1], feat_imp.values[::-1], color="seagreen", edgecolor="black", linewidth=0.4)
axes[2].set_title("Top 10 Feature Importances — RF", fontweight="bold")
axes[2].set_xlabel("Importance")

plt.tight_layout()
plt.savefig("../images/rf_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. XGBoost
XGBoost (Extreme Gradient Boosting) nderton peme vendimi **sekuencialisht** — secila peme korrigjon gabimet e pemes paraardhese (boosting).

**Avantazhet kryesore:**
- Shpesh performon me mire se Random Forest ne dataset-e te pabalancuara
-  rregullon automatikisht peshen e klases minoritare
- Shume i shpejte dhe memory-efficient

**Hyperparametrat kryesore:** , , , 

In [ ]:
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

param_grid_xgb = {
    "n_estimators"    : [100, 200, 300],
    "max_depth"       : [3, 5, 7],
    "learning_rate"   : [0.01, 0.1, 0.2],
    "scale_pos_weight": [scale_pos]
}

grid_xgb = GridSearchCV(
    XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss", verbosity=0),
    param_grid_xgb,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_xgb.fit(X_train, y_train)

print("=" * 50)
print("REZULTATET E GRIDSEARCHCV — XGBOOST")
print("=" * 50)
print(f"Parametrat me te mire : {grid_xgb.best_params_}")
print(f"F1-score me i mire (CV): {grid_xgb.best_score_:.4f}")

In [ ]:
results_xgb = pd.DataFrame(grid_xgb.cv_results_)
results_xgb = results_xgb[["param_n_estimators", "param_max_depth",
                            "param_learning_rate", "mean_test_score", "std_test_score"]]
results_xgb = results_xgb.sort_values("mean_test_score", ascending=False)
results_xgb.columns = ["N Estimators", "Max Depth", "Learning Rate", "F1 mesatar (CV)", "Std"]
print("Top 8 kombinime:")
print(results_xgb.head(8).round(4).to_string(index=False))

### 4.2 Vlerësimi i XGBoost
Vlerësojmë modelin më të mirë XGBoost në **test set** me të njëjtat metrika.

In [ ]:
best_xgb   = grid_xgb.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)
y_prob_xgb = best_xgb.predict_proba(X_test)[:, 1]

acc_xgb  = accuracy_score(y_test, y_pred_xgb)
prec_xgb = precision_score(y_test, y_pred_xgb)
rec_xgb  = recall_score(y_test, y_pred_xgb)
f1_xgb   = f1_score(y_test, y_pred_xgb)
auc_xgb  = roc_auc_score(y_test, y_prob_xgb)

print("=" * 50)
print("METRIKAT — XGBOOST (Test Set)")
print("=" * 50)
print(f"Accuracy  : {acc_xgb:.4f}")
print(f"Precision : {prec_xgb:.4f}")
print(f"Recall    : {rec_xgb:.4f}")
print(f"F1-score  : {f1_xgb:.4f}")
print(f"ROC-AUC   : {auc_xgb:.4f}")
print(f"
{classification_report(y_test, y_pred_xgb, target_names=['Legjitime', 'Mashtruese'])}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

cm_xgb = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm_xgb, annot=True, fmt="d", cmap="Purples", ax=axes[0],
            xticklabels=["Legjitime", "Mashtruese"],
            yticklabels=["Legjitime", "Mashtruese"])
axes[0].set_title("Confusion Matrix — XGBoost", fontweight="bold")
axes[0].set_ylabel("Aktuale")
axes[0].set_xlabel("Te parashikuara")

fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)
axes[1].plot(fpr_xgb, tpr_xgb, color="purple", linewidth=2,
             label=f"XGBoost (AUC = {auc_xgb:.4f})")
axes[1].plot([0, 1], [0, 1], "k--", linewidth=1, label="Random Classifier")
axes[1].set_title("ROC Curve — XGBoost", fontweight="bold")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend()

feat_imp_xgb = pd.Series(best_xgb.feature_importances_, index=feature_names).sort_values(ascending=False).head(10)
axes[2].barh(feat_imp_xgb.index[::-1], feat_imp_xgb.values[::-1], color="purple", edgecolor="black", linewidth=0.4)
axes[2].set_title("Top 10 Feature Importances — XGBoost", fontweight="bold")
axes[2].set_xlabel("Importance")

plt.tight_layout()
plt.savefig("../images/xgb_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()